In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Agent Serving / Deployment Wrapper
# Cell 1 — Deployment config + table checks
# ─────────────────────────────────────────────────────────────

import json
import platform
from datetime import datetime, timezone

import mlflow
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Project / schema config
# ─────────────────────────────────────────────────────────────

GOLD_SCHEMA = "supplysage_gold"

PROJECT_NAME = "SupplySage AI"
AGENT_NAME = "supplysage_supplier_risk_agent"
AGENT_VERSION = "v1_manual_graph"
DEPLOYMENT_VERSION = "v1_delta_serving_wrapper"

# Upstream tables
CHAT_CONTEXT_TABLE = f"{GOLD_SCHEMA}.gold_chat_context_snapshots"
EMBEDDINGS_TABLE = f"{GOLD_SCHEMA}.gold_rag_embeddings"
RETRIEVAL_INDEX_TABLE = f"{GOLD_SCHEMA}.gold_rag_retrieval_index"

# Observability tables from Notebook 28
AGENT_RUN_LOG_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs"
AGENT_MONITORING_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_monitoring_metrics"
AGENT_EVAL_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_eval_results"
AGENT_MONITORING_SUMMARY_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_monitoring_summary"

# New serving/deployment tables
AGENT_SERVING_REQUESTS_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_requests"
AGENT_SERVING_RESPONSES_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_responses"
AGENT_SERVING_HEALTH_TABLE = f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_health"

# MLflow experiment from Notebook 28
MLFLOW_EXPERIMENT_NAME = "/Users/vigya.awasthi@tamu.edu/supplysage_ai_agent_monitoring"

# Runtime defaults
TOP_K_CANDIDATES = 2000
TOP_K_FINAL = 8
ST_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

# Serving defaults
DEFAULT_CLIENT_NAME = "supplysage_ui"
DEFAULT_REQUEST_SOURCE = "notebook_29_serving_test"

# ─────────────────────────────────────────────────────────────
# MLflow setup
# ─────────────────────────────────────────────────────────────

try:
    mlflow.set_tracking_uri("databricks")
except Exception as e:
    print("Could not explicitly set Databricks tracking URI. Continuing.")
    print("Tracking URI warning:", str(e))

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

print("MLflow configured.")
print("Experiment name:", MLFLOW_EXPERIMENT_NAME)
print("Experiment ID:", experiment.experiment_id if experiment else None)
print("Tracking URI:", mlflow.get_tracking_uri())

# ─────────────────────────────────────────────────────────────
# Deployment metadata
# ─────────────────────────────────────────────────────────────

deployment_metadata = {
    "project_name": PROJECT_NAME,
    "agent_name": AGENT_NAME,
    "agent_version": AGENT_VERSION,
    "deployment_version": DEPLOYMENT_VERSION,
    "gold_schema": GOLD_SCHEMA,
    "python_version": platform.python_version(),
    "mlflow_version": mlflow.__version__,
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

print("\nDeployment metadata:")
print(json.dumps(deployment_metadata, indent=2))

# ─────────────────────────────────────────────────────────────
# Required table checks
# ─────────────────────────────────────────────────────────────

required_tables = [
    CHAT_CONTEXT_TABLE,
    EMBEDDINGS_TABLE,
    RETRIEVAL_INDEX_TABLE,
    AGENT_MONITORING_TABLE,
    AGENT_EVAL_TABLE,
    AGENT_MONITORING_SUMMARY_TABLE
]

table_counts = {}

for table_name in required_tables:
    print(f"\nChecking table: {table_name}")

    try:
        row_count = spark.table(table_name).count()
        table_counts[table_name] = row_count
        print(f"Rows: {row_count}")
    except Exception as e:
        raise ValueError(f"Required table check failed for {table_name}: {str(e)}")

assert table_counts[CHAT_CONTEXT_TABLE] > 0, "No supplier snapshots found."
assert table_counts[EMBEDDINGS_TABLE] > 0, "No RAG embeddings found."
assert table_counts[RETRIEVAL_INDEX_TABLE] > 0, "No retrieval index rows found."
assert table_counts[AGENT_MONITORING_TABLE] > 0, "No monitoring metrics found. Run Notebook 28."
assert table_counts[AGENT_EVAL_TABLE] > 0, "No agent eval results found. Run Notebook 28."
assert table_counts[AGENT_MONITORING_SUMMARY_TABLE] > 0, "No monitoring summary found. Run Notebook 28."

print("\nAll deployment prerequisites are available.")
print("Notebook 29 setup complete.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Cell 2
# Recreate supplier risk agent runtime for serving wrapper
# ─────────────────────────────────────────────────────────────

import json
import re
import time
import traceback
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Install / load embedding model
# ─────────────────────────────────────────────────────────────

subprocess.run(
    ["pip", "install", "sentence-transformers", "--quiet"],
    check=True
)

from sentence_transformers import SentenceTransformer

print(f"Loading retrieval model: {ST_MODEL_NAME}")
retrieval_model = SentenceTransformer(ST_MODEL_NAME)

actual_dim = int(
    retrieval_model.get_embedding_dimension()
    if hasattr(retrieval_model, "get_embedding_dimension")
    else retrieval_model.get_sentence_embedding_dimension()
)

assert actual_dim == EMBEDDING_DIM, f"Expected embedding dim {EMBEDDING_DIM}, got {actual_dim}"
print(f"Retrieval model ready. Embedding dim: {actual_dim}")

# ─────────────────────────────────────────────────────────────
# Supplier resolver
# ─────────────────────────────────────────────────────────────

supplier_lookup_pd = (
    spark.table(CHAT_CONTEXT_TABLE)
    .select("supplier_id", "supplier_name")
    .dropDuplicates(["supplier_id"])
    .orderBy("supplier_id")
    .toPandas()
)

supplier_lookup_pd["supplier_id_upper"] = supplier_lookup_pd["supplier_id"].astype(str).str.upper()
supplier_lookup_pd["supplier_name_lower"] = supplier_lookup_pd["supplier_name"].astype(str).str.lower()

print(f"Loaded supplier resolver rows: {len(supplier_lookup_pd)}")

def resolve_supplier_from_question(question: str) -> dict:
    if question is None or len(str(question).strip()) == 0:
        return {
            "supplier_id": None,
            "supplier_name": None,
            "confidence": 0.0,
            "resolution_method": "empty_question",
            "message": "Question is empty."
        }

    question_text = str(question).strip()
    question_lower = question_text.lower()

    # Explicit supplier ID
    supplier_id_match = re.search(
        r"\bSUP[_\- ]?(\d+)\b",
        question_text,
        flags=re.IGNORECASE
    )

    if supplier_id_match:
        supplier_num = supplier_id_match.group(1)
        normalized_supplier_id = f"SUP_{supplier_num}"

        match_pd = supplier_lookup_pd[
            supplier_lookup_pd["supplier_id_upper"] == normalized_supplier_id.upper()
        ]

        if len(match_pd) > 0:
            row = match_pd.iloc[0]
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 1.0,
                "resolution_method": "explicit_supplier_id",
                "message": f"Resolved explicit supplier ID {row['supplier_id']}."
            }

    # Exact supplier name substring
    for _, row in supplier_lookup_pd.iterrows():
        if row["supplier_name_lower"] in question_lower:
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 0.95,
                "resolution_method": "supplier_name_exact_substring",
                "message": f"Resolved supplier name {row['supplier_name']}."
            }

    # First token match, useful for names like FlexPack
    for _, row in supplier_lookup_pd.iterrows():
        first_token = str(row["supplier_name"]).split()[0].lower()

        if len(first_token) >= 4 and first_token in question_lower:
            return {
                "supplier_id": row["supplier_id"],
                "supplier_name": row["supplier_name"],
                "confidence": 0.85,
                "resolution_method": "supplier_name_first_token",
                "message": f"Resolved supplier by first token {first_token}."
            }

    return {
        "supplier_id": None,
        "supplier_name": None,
        "confidence": 0.0,
        "resolution_method": "unresolved",
        "message": "Could not resolve supplier from question."
    }


# ─────────────────────────────────────────────────────────────
# Snapshot helpers
# ─────────────────────────────────────────────────────────────

def parse_json_maybe(value):
    if value is None:
        return None

    if isinstance(value, dict) or isinstance(value, list):
        return value

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return json.loads(value)
    except Exception:
        return None


def extract_line_value(text: str, label: str):
    if text is None:
        return None

    text = str(text)

    pattern = rf"{re.escape(label)}\s*:\s*(.+)"
    match = re.search(pattern, text, flags=re.IGNORECASE)

    if not match:
        return None

    value = match.group(1).strip()

    value = re.split(
        r"\s+\|\s+|Open alerts:|Active external events:|Top affected SKU count:|Risk explanation:",
        value
    )[0].strip()

    return value if value else None


def get_supplier_snapshot(supplier_id: str) -> dict:
    if supplier_id is None:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    supplier_id = str(supplier_id).strip().upper()

    rows = (
        spark.table(CHAT_CONTEXT_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .limit(1)
        .collect()
    )

    if not rows:
        return {
            "ok": False,
            "error": f"No snapshot found for supplier_id={supplier_id}.",
            "supplier_id": supplier_id,
            "snapshot": None
        }

    row = rows[0].asDict(recursive=True)

    chat_context_text = row.get("chat_context_text")
    chat_context_json = parse_json_maybe(row.get("chat_context_json")) or {}

    top_risk_driver = (
        row.get("final_top_risk_driver")
        or row.get("top_risk_driver")
        or chat_context_json.get("top_risk_driver")
        or chat_context_json.get("risk_driver")
        or extract_line_value(chat_context_text, "Top risk driver")
    )

    recommended_action = (
        row.get("final_recommended_action")
        or row.get("recommended_action")
        or chat_context_json.get("recommended_action")
        or chat_context_json.get("action")
        or extract_line_value(chat_context_text, "Recommended action")
    )

    snapshot = {
        "supplier_id": row.get("supplier_id"),
        "supplier_name": row.get("supplier_name"),
        "snapshot_date": row.get("snapshot_date"),
        "snapshot_timestamp": row.get("snapshot_timestamp"),
        "supplier_risk_score": row.get("supplier_risk_score"),
        "risk_band": row.get("risk_band"),
        "score_delta": row.get("score_delta"),
        "criticality_tier": row.get("criticality_tier"),
        "annual_spend": row.get("annual_spend"),
        "mapped_sku_count": row.get("mapped_sku_count"),
        "open_alert_count": row.get("open_alert_count"),
        "critical_or_high_alert_count": row.get("critical_or_high_alert_count"),
        "active_external_event_count": row.get("active_external_event_count"),
        "high_severity_event_count": row.get("high_severity_event_count"),
        "top_affected_sku_count": row.get("top_affected_sku_count"),
        "high_risk_sku_count": row.get("high_risk_sku_count"),
        "latest_external_event_date": row.get("latest_external_event_date"),
        "top_risk_driver": top_risk_driver,
        "recommended_action": recommended_action,
        "top_affected_skus": parse_json_maybe(row.get("top_affected_skus")),
        "active_external_events": parse_json_maybe(row.get("active_external_events")),
        "open_alerts": parse_json_maybe(row.get("open_alerts")),
        "chat_context_text": chat_context_text,
        "chat_context_json": chat_context_json
    }

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "supplier_name": snapshot.get("supplier_name"),
        "snapshot": snapshot
    }


# ─────────────────────────────────────────────────────────────
# Evidence retrieval + cleaning
# ─────────────────────────────────────────────────────────────

def clean_evidence_text(text, max_chars: int = 700) -> str:
    if text is None:
        return ""

    text = str(text)

    if '"raw_text"' in text or "'raw_text'" in text:
        raw_text_pos = text.find('"raw_text"')

        if raw_text_pos == -1:
            raw_text_pos = text.find("'raw_text'")

        prefix = text[:raw_text_pos].strip()

        if len(prefix) > 30:
            text = prefix + " [Raw payload omitted for readability.]"
        else:
            text = "Raw sanctions/compliance payload detected. Full payload omitted for readability."

    text = re.sub(
        r"<\?xml.*",
        "[Raw XML payload omitted for readability.]",
        text,
        flags=re.DOTALL
    )

    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    text = text.replace("| { [Raw payload omitted for readability.]", "| Raw payload omitted for readability.")
    text = text.replace("{ [Raw payload omitted for readability.]", "Raw payload omitted for readability.")

    if len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."

    return text


def retrieve_supplier_evidence(
    supplier_id: str,
    question: str,
    top_k_candidates: int = TOP_K_CANDIDATES,
    top_k_final: int = TOP_K_FINAL
) -> dict:
    if supplier_id is None or len(str(supplier_id).strip()) == 0:
        return {
            "ok": False,
            "error": "supplier_id is required.",
            "supplier_id": supplier_id,
            "evidence": []
        }

    if question is None or len(str(question).strip()) == 0:
        return {
            "ok": False,
            "error": "question is required.",
            "supplier_id": supplier_id,
            "evidence": []
        }

    supplier_id = str(supplier_id).strip().upper()
    question = str(question).strip()

    query_embedding = retrieval_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    query_vector = np.array(query_embedding, dtype=np.float32)

    candidate_pd = (
        spark.table(EMBEDDINGS_TABLE)
        .filter(F.col("supplier_id") == supplier_id)
        .select(
            "chunk_id",
            "original_chunk_id",
            "chunk_type",
            "supplier_id",
            "sku_id",
            "source_name",
            "risk_category",
            "event_date",
            "severity",
            "freshness_weight",
            "source_url",
            "evidence_doc_id",
            "chunk_text",
            "embedding"
        )
        .orderBy(
            F.desc("freshness_weight"),
            F.desc("event_date")
        )
        .limit(top_k_candidates)
        .toPandas()
    )

    if len(candidate_pd) == 0:
        return {
            "ok": False,
            "error": f"No evidence found for supplier {supplier_id}.",
            "supplier_id": supplier_id,
            "candidate_count": 0,
            "evidence": []
        }

    embedding_matrix = np.vstack(
        candidate_pd["embedding"]
        .apply(lambda x: np.array(x, dtype=np.float32))
        .values
    )

    if embedding_matrix.shape[1] != len(query_vector):
        return {
            "ok": False,
            "error": f"Embedding dimension mismatch. Evidence dim={embedding_matrix.shape[1]}, query dim={len(query_vector)}",
            "supplier_id": supplier_id,
            "candidate_count": len(candidate_pd),
            "evidence": []
        }

    candidate_pd["cosine_similarity"] = embedding_matrix @ query_vector

    candidate_pd["freshness_weight"] = pd.to_numeric(
        candidate_pd["freshness_weight"],
        errors="coerce"
    ).fillna(0.5)

    candidate_pd["event_date_parsed"] = pd.to_datetime(
        candidate_pd["event_date"],
        errors="coerce"
    )

    today = pd.Timestamp.today().normalize()

    candidate_pd["days_since_event"] = (
        today - candidate_pd["event_date_parsed"]
    ).dt.days

    candidate_pd["recency_score"] = np.select(
        [
            candidate_pd["event_date_parsed"].isna(),
            candidate_pd["days_since_event"] <= 30,
            candidate_pd["days_since_event"] <= 90,
            candidate_pd["days_since_event"] <= 365,
        ],
        [
            0.35,
            1.00,
            0.85,
            0.65,
        ],
        default=0.25
    )

    severity_map = {
        "critical": 1.00,
        "high": 0.90,
        "medium": 0.55,
        "low": 0.30
    }

    candidate_pd["severity_score"] = (
        candidate_pd["severity"]
        .fillna("unknown")
        .astype(str)
        .str.lower()
        .map(severity_map)
        .fillna(0.40)
    )

    driver_sources = ["internal_risk_engine", "ofac", "cisa", "nws", "eia", "sec", "cpsc"]

    candidate_pd["driver_source_score"] = (
        candidate_pd["source_name"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(driver_sources)
        .astype(float)
    )

    driver_categories = [
        "supplier_risk",
        "sanctions_compliance",
        "cyber",
        "logistics",
        "operational",
        "quality_recall",
        "weather",
        "energy"
    ]

    candidate_pd["driver_category_score"] = (
        candidate_pd["risk_category"]
        .fillna("")
        .astype(str)
        .str.lower()
        .isin(driver_categories)
        .astype(float)
    )

    candidate_pd["business_retrieval_score"] = (
        0.45 * candidate_pd["cosine_similarity"]
        + 0.15 * candidate_pd["freshness_weight"]
        + 0.15 * candidate_pd["severity_score"]
        + 0.10 * candidate_pd["recency_score"]
        + 0.10 * candidate_pd["driver_source_score"]
        + 0.05 * candidate_pd["driver_category_score"]
    )

    ranked_pd = (
        candidate_pd
        .sort_values(
            by=[
                "business_retrieval_score",
                "driver_source_score",
                "severity_score",
                "recency_score",
                "cosine_similarity"
            ],
            ascending=False
        )
        .head(top_k_final)
        .copy()
        .reset_index(drop=True)
    )

    evidence = []

    for idx, row in ranked_pd.iterrows():
        chunk_text = row.get("chunk_text")

        evidence.append({
            "rank": int(idx + 1),
            "chunk_id": row.get("chunk_id"),
            "original_chunk_id": row.get("original_chunk_id"),
            "chunk_type": row.get("chunk_type"),
            "supplier_id": row.get("supplier_id"),
            "sku_id": row.get("sku_id"),
            "source_name": row.get("source_name"),
            "risk_category": row.get("risk_category"),
            "event_date": str(row.get("event_date")) if pd.notna(row.get("event_date")) else None,
            "severity": row.get("severity"),
            "freshness_weight": float(row.get("freshness_weight")) if pd.notna(row.get("freshness_weight")) else None,
            "cosine_similarity": float(row.get("cosine_similarity")) if pd.notna(row.get("cosine_similarity")) else None,
            "business_retrieval_score": float(row.get("business_retrieval_score")) if pd.notna(row.get("business_retrieval_score")) else None,
            "source_url": row.get("source_url"),
            "evidence_doc_id": row.get("evidence_doc_id"),
            "cleaned_evidence_text": clean_evidence_text(chunk_text)
        })

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "question": question,
        "candidate_count": int(len(candidate_pd)),
        "returned_count": int(len(evidence)),
        "evidence": evidence
    }


# ─────────────────────────────────────────────────────────────
# Answer generator
# ─────────────────────────────────────────────────────────────

def generate_supplier_risk_answer(
    question: str,
    snapshot_result: dict,
    cleaned_evidence_result: dict
) -> dict:
    if not snapshot_result.get("ok"):
        return {
            "ok": False,
            "error": snapshot_result.get("error", "Supplier snapshot failed."),
            "answer": None
        }

    if not cleaned_evidence_result.get("ok"):
        return {
            "ok": False,
            "error": cleaned_evidence_result.get("error", "Evidence retrieval failed."),
            "answer": None
        }

    snapshot = snapshot_result["snapshot"]
    snapshot_json = snapshot.get("chat_context_json", {}) or {}
    evidence = cleaned_evidence_result.get("evidence", []) or []

    supplier_id = snapshot["supplier_id"]
    supplier_name = snapshot["supplier_name"]
    risk_score = snapshot["supplier_risk_score"]
    risk_band = snapshot["risk_band"]

    top_driver = (
        snapshot.get("top_risk_driver")
        or snapshot_json.get("top_risk_driver")
        or "Unknown"
    )

    recommended_action = (
        snapshot.get("recommended_action")
        or snapshot_json.get("recommended_action")
        or "Review supplier risk evidence and monitor for changes."
    )

    top_skus = (
        snapshot.get("top_affected_skus")
        or snapshot_json.get("top_affected_skus")
        or []
    )

    active_events = (
        snapshot.get("active_external_events")
        or snapshot_json.get("active_external_events")
        or []
    )

    q_lower = str(question).lower()

    if "high risk" in q_lower and str(risk_band).lower() != "high":
        risk_opening = f"{supplier_name} is currently classified as {risk_band}, not high risk."
    else:
        risk_opening = f"{supplier_name} is currently classified as {risk_band}."

    event_lines = []

    for event in active_events[:5]:
        event_lines.append(
            f"- {event.get('source_name')} | {event.get('risk_category')} | "
            f"{event.get('severity')} | {event.get('event_date')}: "
            f"{event.get('event_title')}"
        )

    if not event_lines:
        event_lines = ["- No active external events found in the supplier snapshot."]

    sku_lines = []

    for sku in top_skus[:6]:
        sku_lines.append(
            f"- {sku.get('canonical_sku_id')}: "
            f"dependency={sku.get('dependency_percent')}, "
            f"stockout risk={sku.get('stockout_risk_score')}, "
            f"primary supplier={sku.get('is_primary_supplier')}, "
            f"alternate supplier={sku.get('alternate_supplier_id')}"
        )

    if not sku_lines:
        sku_lines = ["- No affected SKUs found in the supplier snapshot."]

    evidence_lines = []

    for ev in evidence[:6]:
        evidence_lines.append(
            f"[{ev.get('rank')}] {ev.get('source_name')} | "
            f"{ev.get('risk_category')} | {ev.get('severity')} | "
            f"{ev.get('event_date')} | {ev.get('cleaned_evidence_text')}"
        )

    if not evidence_lines:
        evidence_lines = ["No retrieved evidence available."]

    high_risk_sku_count = snapshot.get("high_risk_sku_count", 0)
    open_alert_count = snapshot.get("open_alert_count", 0)

    if high_risk_sku_count == 0 and open_alert_count == 0:
        operational_interpretation = (
            "This is currently an event-driven supplier monitoring case, "
            "not an immediate stockout or open-alert escalation. "
            "The affected SKUs do not currently show high stockout risk, "
            "so the right action is continued monitoring unless new external events or alerts appear."
        )
    else:
        operational_interpretation = (
            "This supplier needs closer operational review because there are either high-risk SKUs "
            "or open alerts associated with the supplier."
        )

    answer = f"""
Question:
{question.strip()}

Answer:
{risk_opening}

{supplier_name} ({supplier_id}) has a supplier risk score of {risk_score}. The current top risk driver is {top_driver}. The recommended action is: {recommended_action}

Why the supplier is being monitored:
- Active external events: {snapshot.get("active_external_event_count", 0)}
- High-severity external events: {snapshot.get("high_severity_event_count", 0)}
- Latest external event date: {snapshot.get("latest_external_event_date")}
- Affected SKUs: {snapshot.get("top_affected_sku_count", 0)}
- High-risk affected SKUs: {snapshot.get("high_risk_sku_count", 0)}
- Open alerts: {snapshot.get("open_alert_count", 0)}

Main external events:
{chr(10).join(event_lines)}

Top affected SKUs:
{chr(10).join(sku_lines)}

Supporting evidence:
{chr(10).join(evidence_lines)}

Recommendation:
{recommended_action}

Operational interpretation:
{operational_interpretation}
""".strip()

    return {
        "ok": True,
        "error": None,
        "supplier_id": supplier_id,
        "supplier_name": supplier_name,
        "answer": answer,
        "evidence_count": len(evidence),
        "risk_band": risk_band,
        "recommended_action": recommended_action,
        "top_risk_driver": top_driver
    }


# ─────────────────────────────────────────────────────────────
# Graph runner
# ─────────────────────────────────────────────────────────────

def run_supplier_risk_graph(question: str) -> dict:
    run_started_at = datetime.now(timezone.utc)

    state = {
        "question": question,
        "ok": None,
        "error": None,
        "trace": [],
        "run_started_at": run_started_at.isoformat(),
        "run_finished_at": None,
        "duration_seconds": None
    }

    try:
        # Node 1: resolve supplier
        resolver_result = resolve_supplier_from_question(question)

        state["resolver"] = resolver_result
        state["supplier_id"] = resolver_result.get("supplier_id")
        state["supplier_name"] = resolver_result.get("supplier_name")

        state["trace"].append({
            "node": "resolve_supplier",
            "ok": resolver_result.get("supplier_id") is not None,
            "method": resolver_result.get("resolution_method"),
            "confidence": resolver_result.get("confidence"),
            "message": resolver_result.get("message")
        })

        if resolver_result.get("supplier_id") is None:
            state["ok"] = False
            state["error"] = "supplier_unresolved"
            state["final_answer"] = (
                "I could not identify a specific supplier from the question. "
                "Please include a supplier ID like SUP_134 or a supplier name like FlexPack Industries."
            )
            return state

        # Node 2: snapshot
        snapshot_result = get_supplier_snapshot(state["supplier_id"])
        state["snapshot_result"] = snapshot_result

        state["trace"].append({
            "node": "get_supplier_snapshot",
            "ok": snapshot_result.get("ok"),
            "error": snapshot_result.get("error")
        })

        if not snapshot_result.get("ok"):
            state["ok"] = False
            state["error"] = snapshot_result.get("error")
            state["final_answer"] = f"I found supplier {state['supplier_id']}, but could not load its snapshot."
            return state

        # Node 3: retrieve evidence
        evidence_result = retrieve_supplier_evidence(
            supplier_id=state["supplier_id"],
            question=question,
            top_k_candidates=TOP_K_CANDIDATES,
            top_k_final=TOP_K_FINAL
        )

        state["cleaned_evidence_result"] = evidence_result

        state["trace"].append({
            "node": "retrieve_and_clean_supplier_evidence",
            "ok": evidence_result.get("ok"),
            "error": evidence_result.get("error"),
            "candidate_count": evidence_result.get("candidate_count"),
            "returned_count": evidence_result.get("returned_count")
        })

        if not evidence_result.get("ok"):
            state["ok"] = False
            state["error"] = evidence_result.get("error")
            state["final_answer"] = f"I loaded the supplier snapshot for {state.get('supplier_name')}, but could not retrieve supporting evidence."
            return state

        # Node 4: answer
        answer_result = generate_supplier_risk_answer(
            question=question,
            snapshot_result=snapshot_result,
            cleaned_evidence_result=evidence_result
        )

        state["answer_result"] = answer_result

        state["trace"].append({
            "node": "generate_supplier_risk_answer",
            "ok": answer_result.get("ok"),
            "error": answer_result.get("error"),
            "evidence_count": answer_result.get("evidence_count")
        })

        if answer_result.get("ok"):
            state["ok"] = True
            state["error"] = None
            state["final_answer"] = answer_result.get("answer")
            state["risk_band"] = answer_result.get("risk_band")
            state["recommended_action"] = answer_result.get("recommended_action")
            state["top_risk_driver"] = answer_result.get("top_risk_driver")
            state["evidence_count"] = answer_result.get("evidence_count")
        else:
            state["ok"] = False
            state["error"] = answer_result.get("error")
            state["final_answer"] = "I could not generate a supplier risk answer."

    except Exception as e:
        state["ok"] = False
        state["error"] = str(e)
        state["final_answer"] = "The supplier risk graph encountered an unexpected error."
        state["exception_traceback"] = traceback.format_exc()

    finally:
        run_finished_at = datetime.now(timezone.utc)
        state["run_finished_at"] = run_finished_at.isoformat()
        state["duration_seconds"] = (run_finished_at - run_started_at).total_seconds()

    return state


# ─────────────────────────────────────────────────────────────
# Runtime sanity test
# ─────────────────────────────────────────────────────────────

serving_runtime_test_question = """
Why is supplier SUP_134 high risk?
Explain the main external events, affected SKUs, and recommended action.
"""

serving_runtime_test_result = run_supplier_risk_graph(serving_runtime_test_question)

print("Serving runtime test:")
print("ok:", serving_runtime_test_result.get("ok"))
print("error:", serving_runtime_test_result.get("error"))
print("supplier_id:", serving_runtime_test_result.get("supplier_id"))
print("supplier_name:", serving_runtime_test_result.get("supplier_name"))
print("risk_band:", serving_runtime_test_result.get("risk_band"))
print("recommended_action:", serving_runtime_test_result.get("recommended_action"))
print("duration_seconds:", serving_runtime_test_result.get("duration_seconds"))

print("\nTrace:")
for step in serving_runtime_test_result.get("trace", []):
    print(step)

print("\nAnswer preview:")
print(serving_runtime_test_result.get("final_answer")[:1200])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Cell 3
# Serving request/response wrapper
# ─────────────────────────────────────────────────────────────

import uuid
import json
import time
import mlflow
from datetime import datetime, timezone

assert "run_supplier_risk_graph" in globals(), "run_supplier_risk_graph missing. Rerun Cell 2."

def build_error_response(
    request_id: str,
    question: str,
    error_code: str,
    error_message: str,
    started_at,
    client_name: str = DEFAULT_CLIENT_NAME,
    request_source: str = DEFAULT_REQUEST_SOURCE
) -> dict:
    finished_at = datetime.now(timezone.utc)

    return {
        "request_id": request_id,
        "ok": False,
        "error_code": error_code,
        "error_message": error_message,
        "question": question,
        "client_name": client_name,
        "request_source": request_source,
        "supplier_id": None,
        "supplier_name": None,
        "risk_band": None,
        "recommended_action": None,
        "top_risk_driver": None,
        "answer": None,
        "evidence": [],
        "trace": [],
        "metrics": {
            "duration_seconds": (finished_at - started_at).total_seconds(),
            "evidence_count": 0,
            "trace_step_count": 0,
            "answer_char_length": 0,
            "supplier_resolved": 0
        },
        "mlflow_run_id": None,
        "agent_name": AGENT_NAME,
        "agent_version": AGENT_VERSION,
        "deployment_version": DEPLOYMENT_VERSION,
        "started_at": started_at.isoformat(),
        "finished_at": finished_at.isoformat()
    }


def extract_serving_metrics(agent_state: dict) -> dict:
    evidence = agent_state.get("cleaned_evidence_result", {}).get("evidence", []) or []
    trace = agent_state.get("trace") or []
    answer = agent_state.get("final_answer") or ""

    source_names = [
        str(ev.get("source_name"))
        for ev in evidence
        if ev.get("source_name")
    ]

    risk_categories = [
        str(ev.get("risk_category"))
        for ev in evidence
        if ev.get("risk_category")
    ]

    metrics = {
        "duration_seconds": float(agent_state.get("duration_seconds") or 0.0),
        "evidence_count": int(agent_state.get("evidence_count") or len(evidence)),
        "trace_step_count": int(len(trace)),
        "answer_char_length": int(len(answer)),
        "supplier_resolved": int(agent_state.get("supplier_id") is not None),
        "unique_source_count": int(len(set(source_names))),
        "unique_risk_category_count": int(len(set(risk_categories))),
        "retrieval_candidate_count": int(
            agent_state.get("cleaned_evidence_result", {}).get("candidate_count") or 0
        ),
        "retrieval_returned_count": int(
            agent_state.get("cleaned_evidence_result", {}).get("returned_count") or 0
        ),
        "raw_payload_leakage_flag": int(
            "<?xml" in answer.lower()
            or "<sdnlist" in answer.lower()
            or '"raw_text"' in answer.lower()
        ),
        "none_leakage_flag": int(
            "recommended action is: none" in answer.lower()
            or "recommendation:\nnone" in answer.lower()
            or "top risk driver is none" in answer.lower()
        )
    }

    return metrics


if hasattr(mlflow, "trace"):
    @mlflow.trace(name="supplysage_serving_request", span_type="CHAIN")
    def traced_serving_graph(question: str) -> dict:
        return run_supplier_risk_graph(question)
else:
    def traced_serving_graph(question: str) -> dict:
        return run_supplier_risk_graph(question)


def serve_supplier_risk_agent(request_payload: dict) -> dict:
    """
    Deployment-facing request/response function.

    Input contract:
    {
      "question": "...",
      "client_name": "supplysage_ui",
      "request_source": "streamlit_app",
      "request_user": "optional",
      "metadata": {}
    }

    Output contract:
    {
      "request_id": "...",
      "ok": true/false,
      "answer": "...",
      "supplier_id": "...",
      "evidence": [...],
      "metrics": {...},
      "mlflow_run_id": "..."
    }
    """

    started_at = datetime.now(timezone.utc)
    request_id = str(request_payload.get("request_id") or f"req_{uuid.uuid4().hex}")

    question = request_payload.get("question")
    client_name = request_payload.get("client_name") or DEFAULT_CLIENT_NAME
    request_source = request_payload.get("request_source") or DEFAULT_REQUEST_SOURCE
    request_user = request_payload.get("request_user")
    request_metadata = request_payload.get("metadata") or {}

    if question is None or len(str(question).strip()) == 0:
        return build_error_response(
            request_id=request_id,
            question=question,
            error_code="invalid_request",
            error_message="question is required.",
            started_at=started_at,
            client_name=client_name,
            request_source=request_source
        )

    question = str(question).strip()

    with mlflow.start_run(run_name=f"{AGENT_NAME}_{DEPLOYMENT_VERSION}") as run:
        mlflow.set_tag("project_name", PROJECT_NAME)
        mlflow.set_tag("agent_name", AGENT_NAME)
        mlflow.set_tag("agent_version", AGENT_VERSION)
        mlflow.set_tag("deployment_version", DEPLOYMENT_VERSION)
        mlflow.set_tag("run_type", "serving_request")
        mlflow.set_tag("client_name", client_name)
        mlflow.set_tag("request_source", request_source)

        if request_user:
            mlflow.set_tag("request_user", str(request_user))

        mlflow.log_param("request_id", request_id)
        mlflow.log_param("question", question)
        mlflow.log_param("client_name", client_name)
        mlflow.log_param("request_source", request_source)

        agent_state = traced_serving_graph(question)

        metrics = extract_serving_metrics(agent_state)

        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        mlflow.log_param("supplier_id", agent_state.get("supplier_id"))
        mlflow.log_param("supplier_name", agent_state.get("supplier_name"))
        mlflow.log_param("risk_band", agent_state.get("risk_band"))
        mlflow.log_param("recommended_action", agent_state.get("recommended_action"))
        mlflow.log_param("ok", agent_state.get("ok"))
        mlflow.log_param("error", agent_state.get("error"))

        mlflow.log_dict(agent_state.get("trace") or [], "serving/trace.json")
        mlflow.log_dict(agent_state.get("cleaned_evidence_result") or {}, "serving/evidence.json")
        mlflow.log_text(agent_state.get("final_answer") or "", "serving/final_answer.txt")
        mlflow.log_dict(request_metadata, "serving/request_metadata.json")

        finished_at = datetime.now(timezone.utc)

        evidence = (
            agent_state.get("cleaned_evidence_result", {}).get("evidence", [])
            if agent_state.get("cleaned_evidence_result")
            else []
        )

        response = {
            "request_id": request_id,
            "ok": bool(agent_state.get("ok")),
            "error_code": agent_state.get("error"),
            "error_message": agent_state.get("error"),
            "question": question,
            "client_name": client_name,
            "request_source": request_source,
            "request_user": request_user,
            "supplier_id": agent_state.get("supplier_id"),
            "supplier_name": agent_state.get("supplier_name"),
            "risk_band": agent_state.get("risk_band"),
            "recommended_action": agent_state.get("recommended_action"),
            "top_risk_driver": agent_state.get("top_risk_driver"),
            "answer": agent_state.get("final_answer"),
            "evidence": evidence,
            "trace": agent_state.get("trace") or [],
            "metrics": metrics,
            "mlflow_run_id": run.info.run_id,
            "mlflow_experiment_id": run.info.experiment_id,
            "agent_name": AGENT_NAME,
            "agent_version": AGENT_VERSION,
            "deployment_version": DEPLOYMENT_VERSION,
            "started_at": started_at.isoformat(),
            "finished_at": finished_at.isoformat()
        }

        mlflow.log_dict(response, "serving/response_payload.json")

    return response


# Test serving wrapper
test_request_payload = {
    "question": "Why is supplier SUP_134 high risk? Explain the main external events, affected SKUs, and recommended action.",
    "client_name": "supplysage_ui",
    "request_source": "notebook_29_serving_test",
    "request_user": "demo_user",
    "metadata": {
        "ui_page": "supplier_risk_chat",
        "test_case": "known_supplier_assumption_correction"
    }
}

serving_response = serve_supplier_risk_agent(test_request_payload)

print("Serving response:")
print("request_id:", serving_response.get("request_id"))
print("ok:", serving_response.get("ok"))
print("error_code:", serving_response.get("error_code"))
print("supplier_id:", serving_response.get("supplier_id"))
print("supplier_name:", serving_response.get("supplier_name"))
print("risk_band:", serving_response.get("risk_band"))
print("recommended_action:", serving_response.get("recommended_action"))
print("top_risk_driver:", serving_response.get("top_risk_driver"))
print("mlflow_run_id:", serving_response.get("mlflow_run_id"))

print("\nMetrics:")
print(json.dumps(serving_response.get("metrics"), indent=2))

print("\nAnswer preview:")
print(serving_response.get("answer")[:1600])

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Cell 4
# Save serving request + response to Delta
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "serving_response" in globals(), "serving_response missing. Rerun Cell 3."

def safe_json_dumps(obj):
    return json.dumps(obj, default=str, ensure_ascii=False)

# ─────────────────────────────────────────────────────────────
# Request row
# ─────────────────────────────────────────────────────────────

request_row = [{
    "request_id": serving_response.get("request_id"),
    "question": serving_response.get("question"),
    "client_name": serving_response.get("client_name"),
    "request_source": serving_response.get("request_source"),
    "request_user": serving_response.get("request_user"),
    "agent_name": serving_response.get("agent_name"),
    "agent_version": serving_response.get("agent_version"),
    "deployment_version": serving_response.get("deployment_version"),
    "request_metadata_json": safe_json_dumps(test_request_payload.get("metadata", {})),
    "request_payload_json": safe_json_dumps(test_request_payload),
    "requested_at": datetime.fromisoformat(serving_response.get("started_at"))
}]

request_schema = StructType([
    StructField("request_id", StringType(), False),
    StructField("question", StringType(), True),
    StructField("client_name", StringType(), True),
    StructField("request_source", StringType(), True),
    StructField("request_user", StringType(), True),
    StructField("agent_name", StringType(), True),
    StructField("agent_version", StringType(), True),
    StructField("deployment_version", StringType(), True),
    StructField("request_metadata_json", StringType(), True),
    StructField("request_payload_json", StringType(), True),
    StructField("requested_at", TimestampType(), True),
])

request_df = spark.createDataFrame(request_row, schema=request_schema)

(
    request_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("29_agent_serving_deployment_wrapper"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_SERVING_REQUESTS_TABLE)
)

print(f"Saved serving request to: {AGENT_SERVING_REQUESTS_TABLE}")

# ─────────────────────────────────────────────────────────────
# Response row
# ─────────────────────────────────────────────────────────────

metrics = serving_response.get("metrics", {}) or {}

response_row = [{
    "request_id": serving_response.get("request_id"),
    "ok": bool(serving_response.get("ok")),
    "error_code": serving_response.get("error_code"),
    "error_message": serving_response.get("error_message"),
    "supplier_id": serving_response.get("supplier_id"),
    "supplier_name": serving_response.get("supplier_name"),
    "risk_band": serving_response.get("risk_band"),
    "recommended_action": serving_response.get("recommended_action"),
    "top_risk_driver": serving_response.get("top_risk_driver"),
    "answer": serving_response.get("answer"),
    "mlflow_run_id": serving_response.get("mlflow_run_id"),
    "mlflow_experiment_id": serving_response.get("mlflow_experiment_id"),
    "duration_seconds": float(metrics.get("duration_seconds") or 0.0),
    "evidence_count": int(metrics.get("evidence_count") or 0),
    "trace_step_count": int(metrics.get("trace_step_count") or 0),
    "answer_char_length": int(metrics.get("answer_char_length") or 0),
    "supplier_resolved": int(metrics.get("supplier_resolved") or 0),
    "unique_source_count": int(metrics.get("unique_source_count") or 0),
    "unique_risk_category_count": int(metrics.get("unique_risk_category_count") or 0),
    "retrieval_candidate_count": int(metrics.get("retrieval_candidate_count") or 0),
    "retrieval_returned_count": int(metrics.get("retrieval_returned_count") or 0),
    "raw_payload_leakage_flag": int(metrics.get("raw_payload_leakage_flag") or 0),
    "none_leakage_flag": int(metrics.get("none_leakage_flag") or 0),
    "trace_json": safe_json_dumps(serving_response.get("trace")),
    "evidence_json": safe_json_dumps(serving_response.get("evidence")),
    "response_payload_json": safe_json_dumps(serving_response),
    "started_at": datetime.fromisoformat(serving_response.get("started_at")),
    "finished_at": datetime.fromisoformat(serving_response.get("finished_at"))
}]

response_schema = StructType([
    StructField("request_id", StringType(), False),
    StructField("ok", BooleanType(), True),
    StructField("error_code", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("supplier_id", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("risk_band", StringType(), True),
    StructField("recommended_action", StringType(), True),
    StructField("top_risk_driver", StringType(), True),
    StructField("answer", StringType(), True),
    StructField("mlflow_run_id", StringType(), True),
    StructField("mlflow_experiment_id", StringType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("evidence_count", IntegerType(), True),
    StructField("trace_step_count", IntegerType(), True),
    StructField("answer_char_length", IntegerType(), True),
    StructField("supplier_resolved", IntegerType(), True),
    StructField("unique_source_count", IntegerType(), True),
    StructField("unique_risk_category_count", IntegerType(), True),
    StructField("retrieval_candidate_count", IntegerType(), True),
    StructField("retrieval_returned_count", IntegerType(), True),
    StructField("raw_payload_leakage_flag", IntegerType(), True),
    StructField("none_leakage_flag", IntegerType(), True),
    StructField("trace_json", StringType(), True),
    StructField("evidence_json", StringType(), True),
    StructField("response_payload_json", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("finished_at", TimestampType(), True),
])

response_df = spark.createDataFrame(response_row, schema=response_schema)

(
    response_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("29_agent_serving_deployment_wrapper"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_SERVING_RESPONSES_TABLE)
)

print(f"Saved serving response to: {AGENT_SERVING_RESPONSES_TABLE}")

# ─────────────────────────────────────────────────────────────
# Preview joined audit log
# ─────────────────────────────────────────────────────────────

display(
    spark.table(AGENT_SERVING_REQUESTS_TABLE).alias("req")
    .join(
        spark.table(AGENT_SERVING_RESPONSES_TABLE).alias("res"),
        on="request_id",
        how="inner"
    )
    .select(
        "request_id",
        "question",
        "client_name",
        "request_source",
        "request_user",
        "ok",
        "supplier_id",
        "supplier_name",
        "risk_band",
        "recommended_action",
        "top_risk_driver",
        "duration_seconds",
        "evidence_count",
        "unique_source_count",
        "raw_payload_leakage_flag",
        "none_leakage_flag",
        "mlflow_run_id",
        "requested_at",
        "finished_at"
    )
    .orderBy(F.desc("requested_at"))
    .limit(10)
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Cell 5
# Serving health check
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "serve_supplier_risk_agent" in globals(), "serve_supplier_risk_agent missing. Rerun Cell 3."

health_started_at = datetime.now(timezone.utc)

health_checks = []

def add_health_check(check_name, ok, details=None, error=None):
    health_checks.append({
        "check_name": check_name,
        "ok": bool(ok),
        "details": details,
        "error": error
    })

# ─────────────────────────────────────────────────────────────
# Check 1: required tables exist and are non-empty
# ─────────────────────────────────────────────────────────────

required_health_tables = [
    CHAT_CONTEXT_TABLE,
    EMBEDDINGS_TABLE,
    RETRIEVAL_INDEX_TABLE,
    AGENT_MONITORING_TABLE,
    AGENT_EVAL_TABLE,
    AGENT_SERVING_REQUESTS_TABLE,
    AGENT_SERVING_RESPONSES_TABLE
]

for table_name in required_health_tables:
    try:
        row_count = spark.table(table_name).count()
        add_health_check(
            check_name=f"table_non_empty::{table_name}",
            ok=row_count > 0,
            details=json.dumps({"row_count": row_count})
        )
    except Exception as e:
        add_health_check(
            check_name=f"table_non_empty::{table_name}",
            ok=False,
            details=None,
            error=str(e)
        )

# ─────────────────────────────────────────────────────────────
# Check 2: model loaded
# ─────────────────────────────────────────────────────────────

try:
    model_dim = int(
        retrieval_model.get_embedding_dimension()
        if hasattr(retrieval_model, "get_embedding_dimension")
        else retrieval_model.get_sentence_embedding_dimension()
    )

    add_health_check(
        check_name="embedding_model_loaded",
        ok=model_dim == EMBEDDING_DIM,
        details=json.dumps({
            "model_name": ST_MODEL_NAME,
            "embedding_dim": model_dim
        })
    )
except Exception as e:
    add_health_check(
        check_name="embedding_model_loaded",
        ok=False,
        details=None,
        error=str(e)
    )

# ─────────────────────────────────────────────────────────────
# Check 3: happy-path serving request
# ─────────────────────────────────────────────────────────────

try:
    health_test_response = serve_supplier_risk_agent({
        "question": "Why is supplier SUP_134 high risk? Explain the main external events and recommended action.",
        "client_name": "health_check",
        "request_source": "notebook_29_health_check",
        "request_user": "system",
        "metadata": {
            "check_type": "happy_path_known_supplier"
        }
    })

    add_health_check(
        check_name="serving_happy_path_known_supplier",
        ok=bool(health_test_response.get("ok"))
            and health_test_response.get("supplier_id") == "SUP_134"
            and health_test_response.get("recommended_action") is not None
            and health_test_response.get("metrics", {}).get("raw_payload_leakage_flag") == 0
            and health_test_response.get("metrics", {}).get("none_leakage_flag") == 0,
        details=json.dumps({
            "request_id": health_test_response.get("request_id"),
            "supplier_id": health_test_response.get("supplier_id"),
            "recommended_action": health_test_response.get("recommended_action"),
            "duration_seconds": health_test_response.get("metrics", {}).get("duration_seconds"),
            "mlflow_run_id": health_test_response.get("mlflow_run_id")
        }, default=str)
    )

except Exception as e:
    add_health_check(
        check_name="serving_happy_path_known_supplier",
        ok=False,
        details=None,
        error=str(e)
    )

# ─────────────────────────────────────────────────────────────
# Check 4: unresolved supplier behavior
# ─────────────────────────────────────────────────────────────

try:
    unresolved_response = serve_supplier_risk_agent({
        "question": "Which supplier should I monitor today?",
        "client_name": "health_check",
        "request_source": "notebook_29_health_check",
        "request_user": "system",
        "metadata": {
            "check_type": "unresolved_supplier"
        }
    })

    unresolved_answer = unresolved_response.get("answer") or ""

    add_health_check(
        check_name="serving_unresolved_supplier_guardrail",
        ok=bool(unresolved_response.get("ok")) is False
            and unresolved_response.get("supplier_id") is None
            and (
                "specific supplier" in unresolved_answer.lower()
                or "supplier id" in unresolved_answer.lower()
                or "could not identify" in unresolved_answer.lower()
            ),
        details=json.dumps({
            "request_id": unresolved_response.get("request_id"),
            "ok": unresolved_response.get("ok"),
            "error_code": unresolved_response.get("error_code"),
            "mlflow_run_id": unresolved_response.get("mlflow_run_id")
        }, default=str)
    )

except Exception as e:
    add_health_check(
        check_name="serving_unresolved_supplier_guardrail",
        ok=False,
        details=None,
        error=str(e)
    )

# ─────────────────────────────────────────────────────────────
# Aggregate health
# ─────────────────────────────────────────────────────────────

health_finished_at = datetime.now(timezone.utc)

total_checks = len(health_checks)
passed_checks = sum(1 for c in health_checks if c["ok"])
failed_checks = total_checks - passed_checks

overall_healthy = failed_checks == 0

health_run_id = f"health_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"

health_rows = []

for c in health_checks:
    health_rows.append({
        "health_run_id": health_run_id,
        "project_name": PROJECT_NAME,
        "agent_name": AGENT_NAME,
        "agent_version": AGENT_VERSION,
        "deployment_version": DEPLOYMENT_VERSION,
        "check_name": c["check_name"],
        "check_ok": bool(c["ok"]),
        "check_details_json": c.get("details"),
        "check_error": c.get("error"),
        "overall_healthy": bool(overall_healthy),
        "total_checks": int(total_checks),
        "passed_checks": int(passed_checks),
        "failed_checks": int(failed_checks),
        "health_duration_seconds": float((health_finished_at - health_started_at).total_seconds()),
        "health_started_at": health_started_at,
        "health_finished_at": health_finished_at
    })

health_schema = StructType([
    StructField("health_run_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("agent_name", StringType(), True),
    StructField("agent_version", StringType(), True),
    StructField("deployment_version", StringType(), True),
    StructField("check_name", StringType(), True),
    StructField("check_ok", BooleanType(), True),
    StructField("check_details_json", StringType(), True),
    StructField("check_error", StringType(), True),
    StructField("overall_healthy", BooleanType(), True),
    StructField("total_checks", IntegerType(), True),
    StructField("passed_checks", IntegerType(), True),
    StructField("failed_checks", IntegerType(), True),
    StructField("health_duration_seconds", DoubleType(), True),
    StructField("health_started_at", TimestampType(), True),
    StructField("health_finished_at", TimestampType(), True),
])

health_df = spark.createDataFrame(health_rows, schema=health_schema)

(
    health_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("29_agent_serving_deployment_wrapper"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_SERVING_HEALTH_TABLE)
)

print(f"Saved serving health check to: {AGENT_SERVING_HEALTH_TABLE}")
print("health_run_id:", health_run_id)
print("overall_healthy:", overall_healthy)
print("passed_checks:", passed_checks)
print("failed_checks:", failed_checks)

display(
    spark.table(AGENT_SERVING_HEALTH_TABLE)
    .filter(F.col("health_run_id") == health_run_id)
    .select(
        "health_run_id",
        "check_name",
        "check_ok",
        "check_error",
        "overall_healthy",
        "passed_checks",
        "failed_checks",
        "health_duration_seconds"
    )
    .orderBy("check_name")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Replacement Cell 6
# Create serving deployment summary table
# Fixed:
# - agent_name / agent_version / deployment_version come from requests table
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F

AGENT_SERVING_SUMMARY_TABLE = "supplysage_gold.gold_supplier_risk_agent_serving_summary"

requests_df = spark.table(AGENT_SERVING_REQUESTS_TABLE)
responses_df = spark.table(AGENT_SERVING_RESPONSES_TABLE)
health_df = spark.table(AGENT_SERVING_HEALTH_TABLE)

# Join request metadata with response metrics
joined_serving_df = (
    responses_df.alias("res")
    .join(
        requests_df.alias("req"),
        on="request_id",
        how="left"
    )
)

# Request/response serving metrics
serving_summary = (
    joined_serving_df
    .groupBy(
        F.col("req.agent_name").alias("agent_name"),
        F.col("req.agent_version").alias("agent_version"),
        F.col("req.deployment_version").alias("deployment_version")
    )
    .agg(
        F.count("*").alias("total_serving_requests"),
        F.sum(F.when(F.col("res.ok") == True, 1).otherwise(0)).alias("successful_serving_requests"),
        F.sum(F.when(F.col("res.ok") == False, 1).otherwise(0)).alias("failed_serving_requests"),
        F.avg(F.col("res.duration_seconds")).alias("avg_duration_seconds"),
        F.expr("percentile_approx(duration_seconds, 0.5)").alias("p50_duration_seconds"),
        F.expr("percentile_approx(duration_seconds, 0.95)").alias("p95_duration_seconds"),
        F.avg(F.col("res.evidence_count")).alias("avg_evidence_count"),
        F.avg(F.col("res.unique_source_count")).alias("avg_unique_source_count"),
        F.sum(F.col("res.raw_payload_leakage_flag")).alias("raw_payload_leakage_count"),
        F.sum(F.col("res.none_leakage_flag")).alias("none_leakage_count"),
        F.countDistinct(F.col("res.supplier_id")).alias("distinct_suppliers_served"),
        F.max(F.col("res.finished_at")).alias("latest_serving_finished_at")
    )
)

# Latest health result
latest_health_run_id = (
    health_df
    .orderBy(F.desc("health_finished_at"))
    .select("health_run_id")
    .limit(1)
    .collect()[0]["health_run_id"]
)

latest_health_summary = (
    health_df
    .filter(F.col("health_run_id") == latest_health_run_id)
    .groupBy("health_run_id")
    .agg(
        F.max("overall_healthy").alias("latest_overall_healthy"),
        F.max("total_checks").alias("latest_total_checks"),
        F.max("passed_checks").alias("latest_passed_checks"),
        F.max("failed_checks").alias("latest_failed_checks"),
        F.max("health_duration_seconds").alias("latest_health_duration_seconds"),
        F.max("health_finished_at").alias("latest_health_finished_at")
    )
    .withColumn("join_key", F.lit(1))
)

summary_df = (
    serving_summary
    .withColumn("join_key", F.lit(1))
    .join(latest_health_summary, on="join_key", how="left")
    .drop("join_key")
    .withColumn(
        "serving_success_rate",
        F.when(
            F.col("total_serving_requests") > 0,
            F.col("successful_serving_requests") / F.col("total_serving_requests")
        ).otherwise(F.lit(None))
    )
    .withColumn("summary_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("29_agent_serving_deployment_wrapper"))
)

(
    summary_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(AGENT_SERVING_SUMMARY_TABLE)
)

print(f"Saved serving summary to: {AGENT_SERVING_SUMMARY_TABLE}")

display(
    spark.table(AGENT_SERVING_SUMMARY_TABLE)
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 29 — Cell 7
# Create deployment manifest for UI / API handoff
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    TimestampType
)

AGENT_DEPLOYMENT_MANIFEST_TABLE = "supplysage_gold.gold_supplier_risk_agent_deployment_manifest"
AGENT_SERVING_SUMMARY_TABLE = "supplysage_gold.gold_supplier_risk_agent_serving_summary"

manifest_id = f"manifest_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"

# Latest serving summary
serving_summary_rows = (
    spark.table(AGENT_SERVING_SUMMARY_TABLE)
    .orderBy(F.desc("summary_created_at"))
    .limit(1)
    .collect()
)

assert len(serving_summary_rows) > 0, "No serving summary found. Rerun Cell 6."

latest_serving_summary = serving_summary_rows[0].asDict(recursive=True)

# Request contract for UI / API style calls
request_contract = {
    "function_name": "serve_supplier_risk_agent",
    "input_payload": {
        "question": "string, required",
        "client_name": "string, optional, default=supplysage_ui",
        "request_source": "string, optional, example=streamlit_app",
        "request_user": "string, optional",
        "metadata": "object, optional"
    },
    "example_request": {
        "question": "Why is supplier SUP_134 high risk? Explain the main external events and recommended action.",
        "client_name": "supplysage_ui",
        "request_source": "streamlit_app",
        "request_user": "demo_user",
        "metadata": {
            "ui_page": "supplier_risk_chat"
        }
    }
}

# Response contract for UI rendering
response_contract = {
    "output_payload": {
        "request_id": "string",
        "ok": "boolean",
        "error_code": "string or null",
        "error_message": "string or null",
        "question": "string",
        "supplier_id": "string or null",
        "supplier_name": "string or null",
        "risk_band": "string or null",
        "recommended_action": "string or null",
        "top_risk_driver": "string or null",
        "answer": "string",
        "evidence": "array of ranked evidence objects",
        "trace": "array of node-level execution steps",
        "metrics": {
            "duration_seconds": "float",
            "evidence_count": "int",
            "trace_step_count": "int",
            "answer_char_length": "int",
            "supplier_resolved": "int",
            "unique_source_count": "int",
            "unique_risk_category_count": "int",
            "retrieval_candidate_count": "int",
            "retrieval_returned_count": "int",
            "raw_payload_leakage_flag": "int",
            "none_leakage_flag": "int"
        },
        "mlflow_run_id": "string",
        "agent_version": "string",
        "deployment_version": "string"
    }
}

# Tables created by this deployment wrapper
deployment_tables = {
    "request_table": AGENT_SERVING_REQUESTS_TABLE,
    "response_table": AGENT_SERVING_RESPONSES_TABLE,
    "health_table": AGENT_SERVING_HEALTH_TABLE,
    "summary_table": AGENT_SERVING_SUMMARY_TABLE,
    "manifest_table": AGENT_DEPLOYMENT_MANIFEST_TABLE,
    "monitoring_table": AGENT_MONITORING_TABLE,
    "evaluation_table": AGENT_EVAL_TABLE
}

# UI handoff notes
ui_handoff = {
    "recommended_ui_pages": [
        {
            "page_name": "Supplier Risk Chat",
            "uses": [
                "serve_supplier_risk_agent(request_payload)",
                AGENT_SERVING_REQUESTS_TABLE,
                AGENT_SERVING_RESPONSES_TABLE
            ]
        },
        {
            "page_name": "Serving Health",
            "uses": [
                AGENT_SERVING_HEALTH_TABLE,
                AGENT_SERVING_SUMMARY_TABLE
            ]
        },
        {
            "page_name": "Agent Observability",
            "uses": [
                AGENT_MONITORING_TABLE,
                AGENT_EVAL_TABLE,
                AGENT_MONITORING_SUMMARY_TABLE
            ]
        }
    ],
    "deployment_status": "ready_for_ui_integration",
    "notes": [
        "This wrapper is dependency-light and Databricks-notebook compatible.",
        "Every serving call writes to MLflow and can be persisted to Delta.",
        "The UI should call the wrapper with a question and render answer, evidence, trace, and metrics."
    ]
}

manifest_row = [{
    "manifest_id": manifest_id,
    "project_name": PROJECT_NAME,
    "agent_name": AGENT_NAME,
    "agent_version": AGENT_VERSION,
    "deployment_version": DEPLOYMENT_VERSION,
    "deployment_status": "ready_for_ui_integration",
    "overall_healthy": bool(latest_serving_summary.get("latest_overall_healthy")),
    "serving_success_rate": str(latest_serving_summary.get("serving_success_rate")),
    "latest_health_run_id": latest_serving_summary.get("health_run_id"),
    "request_contract_json": json.dumps(request_contract, indent=2, default=str),
    "response_contract_json": json.dumps(response_contract, indent=2, default=str),
    "deployment_tables_json": json.dumps(deployment_tables, indent=2, default=str),
    "ui_handoff_json": json.dumps(ui_handoff, indent=2, default=str),
    "created_at": datetime.now(timezone.utc)
}]

manifest_schema = StructType([
    StructField("manifest_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("agent_name", StringType(), True),
    StructField("agent_version", StringType(), True),
    StructField("deployment_version", StringType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("overall_healthy", BooleanType(), True),
    StructField("serving_success_rate", StringType(), True),
    StructField("latest_health_run_id", StringType(), True),
    StructField("request_contract_json", StringType(), True),
    StructField("response_contract_json", StringType(), True),
    StructField("deployment_tables_json", StringType(), True),
    StructField("ui_handoff_json", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

manifest_df = spark.createDataFrame(manifest_row, schema=manifest_schema)

(
    manifest_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit("29_agent_serving_deployment_wrapper"))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(AGENT_DEPLOYMENT_MANIFEST_TABLE)
)

print(f"Saved deployment manifest to: {AGENT_DEPLOYMENT_MANIFEST_TABLE}")
print("manifest_id:", manifest_id)
print("deployment_status: ready_for_ui_integration")
print("overall_healthy:", bool(latest_serving_summary.get("latest_overall_healthy")))
print("serving_success_rate:", latest_serving_summary.get("serving_success_rate"))

display(
    spark.table(AGENT_DEPLOYMENT_MANIFEST_TABLE)
    .filter(F.col("manifest_id") == manifest_id)
    .select(
        "manifest_id",
        "project_name",
        "agent_name",
        "agent_version",
        "deployment_version",
        "deployment_status",
        "overall_healthy",
        "serving_success_rate",
        "created_at"
    )
)